In [6]:
from utils import MOVIELENS_RATINGS_PATH, train_test_split_ratings, rmse_on_test
from slim import SLIM

R_train, user_ids, movie_ids, test_u, test_i, test_ratings = train_test_split_ratings(
    MOVIELENS_RATINGS_PATH,
    test_size=0.2,
    random_state=42,
)
print(f"train nnz={R_train.nnz}, test pairs={len(test_ratings)}")

train nnz=80668, test pairs=20168


# SLIM

In [2]:
model = SLIM(l1_reg=1e-1, l2_reg=1e-1, max_iter=100)
model.fit(R_train)

test_rmse = rmse_on_test(model, R_train, test_u, test_i, test_ratings)
print(f"test RMSE: {test_rmse:.4f}")

  0%|          | 0/9724 [00:00<?, ?it/s]

100%|██████████| 9724/9724 [00:58<00:00, 166.80it/s]


test RMSE: 3.0190


In [3]:
# для сравнения: RMSE на train (обычно ниже test)
train_u, train_i = R_train.nonzero()
train_rmse = rmse_on_test(model, R_train, train_u, train_i, R_train.data)
print(f"train RMSE: {train_rmse:.4f}")

test_rmse = rmse_on_test(model, R_train, test_u, test_i, test_ratings)
print(f"test RMSE: {test_rmse:.4f}")

train RMSE: 2.7188
test RMSE: 3.0190


In [5]:
from reference_slim import ReferenceSLIM, is_reference_slim_available
from utils import compare_models_rmse

if not is_reference_slim_available():
    raise RuntimeError("Сначала соберите эталон: bash build_reference_slim.sh")

ref_model = ReferenceSLIM(l1_reg=1e-1, l2_reg=1e-1, nthreads=2, max_iter=100)
ref_model.fit(R_train)

Using Coordinate Descent! 
Learning takes 61.640 secs.


In [7]:
compare_models_rmse(
    {"own_slim": model, "reference_slim": ref_model},
    R_train,
    test_u,
    test_i,
    test_ratings,
)

own_slim          3.019026
reference_slim    3.330592
Name: test_rmse, dtype: float64

# ALS

In [2]:
from als import ALS

In [3]:
model = ALS(n_factors=50, reg=1e-1, n_iter=100)
model.fit(R_train)

test_rmse = rmse_on_test(model, R_train, test_u, test_i, test_ratings)
print(f"test RMSE: {test_rmse:.4f}")

ALS:   0%|          | 0/100 [00:00<?, ?it/s]

ALS: 100%|██████████| 100/100 [00:31<00:00,  3.20it/s]

test RMSE: 1.2657


In [4]:
from reference_als import ImplicitALS

model = ImplicitALS(factors=50, regularization=1e-1, iterations=100)
model.fit(R_train)

/Users/hq-q05j3pd44y/Documents/vscode/spring-26/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 100/100 [00:02<00:00, 34.55it/s]


In [5]:
rmse_on_test(model, R_train, test_u, test_i, test_ratings)

3.141171390745033